# 📈 InsightForge AI — Trend & Time Series Forecasting
Provides automated trend identification and baseline linear / polynomial forecasting for numerical features over time.


In [ ]:
%pip install pandas numpy plotly scikit-learn
dbutils.library.restartPython()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

def generate_trend_forecast(df: pd.DataFrame, date_col: str, target_col: str, periods_ahead: int = 6) -> dict:
    """
    Detects date and numerical trends, fits linear projection, and returns forecast plot and growth metrics.
    """
    df_clean = df[[date_col, target_col]].dropna().copy()
    df_clean[date_col] = pd.to_datetime(df_clean[date_col])
    df_clean = df_clean.sort_values(date_col)
    
    # Feature engineering for trend
    df_clean["time_idx"] = np.arange(len(df_clean))
    X = df_clean[["time_idx"]].values
    y = df_clean[target_col].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    future_idx = np.arange(len(df_clean), len(df_clean) + periods_ahead).reshape(-1, 1)
    future_preds = model.predict(future_idx)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_clean[date_col], y=y, mode="lines+markers", name="Historical Actuals"))
    
    # Future dates
    last_date = df_clean[date_col].iloc[-1]
    future_dates = pd.date_range(start=last_date, periods=periods_ahead+1, freq='M')[1:]
    fig.add_trace(go.Scatter(x=future_dates, y=future_preds, mode="lines+markers", line=dict(dash="dash", color="red"), name="Projected Forecast"))
    
    fig.update_layout(title=f"Forecast Projection: {target_col} over {date_col}", template="plotly_white")
    
    slope = model.coef_[0]
    trend_direction = "Upward" if slope > 0 else "Downward" if slope < 0 else "Flat"
    
    return {
        "fig": fig,
        "trend_direction": trend_direction,
        "slope_per_period": round(slope, 3),
        "forecast_values": [round(v, 2) for v in future_preds]
    }
